# End of week 1 exercise

A personal **tech tutor**: paste a question or code snippet, pick an audience level, and get explanations from both **GPT-4o-mini** (streaming) and **Llama 3.2** (Ollama), then a short comparison of the two answers.

Requires: `OPENAI_API_KEY` in `.env`, and Ollama running locally with `ollama pull llama3.2`.

In [ ]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-proj-") and len(api_key) > 10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key — see the troubleshooting notebook")

openai = OpenAI()
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
# audience: "beginner" | "intermediate" | "java-developer" | "interview"
AUDIENCE = "java-developer"

AUDIENCE_INSTRUCTIONS = {
    "beginner": (
        "Explain like to a smart beginner. Use simple language, short analogies, "
        "and avoid jargon unless you define it. Show a tiny concrete example."
    ),
    "intermediate": (
        "Assume solid Python basics. Be precise about semantics, edge cases, and "
        "when you'd use this pattern in real code. Keep it concise."
    ),
    "java-developer": (
        "Assume solid Python and Java basics. Use java terms and examples to explain if it is possible. Be precise about semantics, edge cases, and "
        "when you'd use this pattern in real code. Keep it concise. Show a tiny concrete example in both languages."
    ),
    "interview": (
        "Answer as for a coding interview. Cover what it does, time/space notes if relevant, "
        "common pitfalls, and a crisp example. Prefer clarity over fluff."
    ),
}

system_prompt = f"""You are a helpful IT technical tutor for Python, Java, software engineering, data science, and LLMs.
Respond in markdown (no outer code fence wrapping the whole answer).
Structure your answer as:
1. What it does
2. Why / when you'd use it
3. Step-by-step
4. Pitfalls
5. Tiny example

Audience guidance: {AUDIENCE_INSTRUCTIONS[AUDIENCE]}
"""

def build_messages(q: str) -> list[dict]:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Please explain the following technical question in detail:\n\n{q}"},
    ]

messages = build_messages(question)
print(f"Audience: {AUDIENCE}")

In [ ]:
# Get gpt-4o-mini to answer, with streaming

def ask_gpt(messages, model=MODEL_GPT):
    stream = openai.chat.completions.create(model=model, messages=messages, stream=True)
    response = ""
    display(Markdown(f"### {model}"))
    handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=handle.display_id)
    return response

gpt_answer = ask_gpt(messages)

In [ ]:
# Get Llama 3.2 to answer

def ask_llama(messages, model=MODEL_LLAMA):
    response = ollama.chat.completions.create(model=model, messages=messages)
    reply = response.choices[0].message.content
    display(Markdown(f"### {model}\n\n{reply}"))
    return reply

llama_answer = ask_llama(messages)

In [ ]:
# Compare the two answers (GPT summarizes where they agree / disagree)

compare_messages = [
    {
        "role": "system",
        "content": (
            "You compare two technical explanations of the same question. "
            "Be concise. Use markdown with sections: Agree, Differ, Prefer which and why."
        ),
    },
    {
        "role": "user",
        "content": f"""Question:
{question}

--- GPT ({MODEL_GPT}) ---
{gpt_answer}

--- Llama ({MODEL_LLAMA}) ---
{llama_answer}

Summarize where they agree, where they differ, and which answer is more useful for a {AUDIENCE} audience.""",
    },
]

_ = ask_gpt(compare_messages)

### Try it yourself

Change `AUDIENCE` to `"intermediate"`, `"java-developer"` or `"interview"`, or overwrite `question`, then re-run from the audience cell downward.

Optional: use `question = input("Your technical question: ")` for interactive prompts.
